# Week1_ex3_zero_tangential_h - Region Walls Switched to Zero Tangential H

Reruns the force-vs-position sweep from `Week1_ex3.ipynb` with the same 50% `Region` padding but with a Zero Tangential H Field boundary on all six `Region` faces, to test the mechanism behind Hypothesis 2 in `Week1_ex3_analysis.ipynb`. The default wall (no boundary assigned) keeps flux from crossing it (B·n = 0) and pushes the magnet away; a zero tangential H wall makes flux enter it straight on, like an infinitely permeable wall, and should pull the magnet toward it instead. If the original `Force_z` curve is a wall image force, its sign should flip at every position.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt

In [ ]:
## Ansys Electronics Desktop (AEDT) interface -- fresh project, not attaching to the original ##

DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)
DT.disable_autosave()

sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

# Separate project name so the solved original project is left untouched
proj_name = "Week1_ex3_zeroHt"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
os.makedirs(dir, exist_ok=True)

desi_name = "Week1_ex3_zeroHt"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Create geometry ##

M3D.modeler.model_units = "cm"

# Register magnet position as a design variable
M3D["magnet_z0"] = "-1cm"

# Create the magnet
center = [0, 0, "magnet_z0"]
origin = [0, 1.3, "magnet_z0"]
height = 2
magnet = M3D.modeler.create_polyhedron(orientation=None, center=center, origin=origin, height=height,
                                        num_sides=36, name="Magnet", material=None)

# Create coil terminal
origin = [0, 1.5, -2.5]
sizes = [1, 5]
coil_terminal = M3D.modeler.create_rectangle(orientation="YZ", origin=origin, sizes=sizes,
                                              name="Coil_Terminal", material=None, is_covered=True)

# Create coil by sweeping the terminal around the axis
coil = coil_terminal.clone()
M3D.modeler.sweep_around_axis(assignment=coil, axis="Z", sweep_angle=360, draft_angle=0, number_of_segments=0)
coil.name = "Coil"

# Create simulation region
region = M3D.modeler.create_region(pad_value=50, pad_type='Percentage Offset', name='Region')

In [ ]:
## Assign boundary on the region faces ##

# assign_zero_tangential_h_field() only accepts Eddy Current in this pyaedt version, so the raw call is used here.
# Face ids are read straight from the editor -- region.faces can come back empty right after create_region().
oEditor = M3D.modeler.oeditor
face_ids = [int(i) for i in oEditor.GetFaceIDs("Region")]
if len(face_ids) != 6:
    raise RuntimeError(f"Expected 6 Region faces, got {len(face_ids)}")

oModule = oDesign.GetModule("BoundarySetup")
oModule.AssignZeroTangentialHField(
	[
		"NAME:ZeroTangentialHField1",
		"Faces:=", face_ids
	])


In [ ]:
## Assign materials ##

magnet_material = M3D.materials.duplicate_material(material="NdFe35", name="Magnet_Material", properties=None)
M3D.assign_material(assignment=magnet, material=magnet_material.name)
magnet_material.set_magnetic_coercivity(value=-890000, x=0, y=0, z=1)

M3D.assign_material(assignment=coil, material="copper")

In [ ]:
## Assign force calculation (force acting on the magnet) ##

# Force is computed and cached at solve time, so it must be assigned here, before running the analysis;
# adding it afterward would require re-solving.
force = M3D.assign_force(assignment="Magnet", is_virtual=True, force_name="Force_on_Magnet")

In [ ]:
## Create and configure analysis setup ##

my_setup = M3D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 6
my_setup.props["PercentError"] = 1
my_setup.update()

In [ ]:
## Assign mesh operations ##

M3D.mesh.assign_length_mesh(assignment=coil, inside_selection=True, maximum_length=1,
                             maximum_elements=700, name="coil_mesh")
M3D.mesh.assign_length_mesh(assignment=magnet, inside_selection=True, maximum_length=1,
                             maximum_elements=400, name="magnet_mesh")

In [ ]:
## Set up and run parametric sweep (with Save Fields enabled) ##

sweep = M3D.parametrics.add(variable="magnet_z0", start_point="-5cm", end_point="3cm",
                             step="0.5cm", variation_type="LinearStep", name="PosSweep")

sweep.props["ProdOptiSetupDataV2"]["SaveFields"] = True
sweep.update()

sweep.analyze()

In [ ]:
## (Check) List available force-related quantities ##

# Confirms the exact quantity name (e.g. Force_on_Magnet.Force_z) before referencing it in get_solution_data() below
available = M3D.post.available_report_quantities(report_category="Fields")
print([q for q in available if "force" in q.lower()])

In [ ]:
## Extract result data (force acting on the magnet) ##
## Match the expressions value to the exact name confirmed in the cell above ##

data = M3D.post.get_solution_data(
    expressions="Force_on_Magnet.Force_z",
    setup_sweep_name="Setup1 : LastAdaptive",
    primary_sweep_variable="magnet_z0",
    variations={"magnet_z0": ["All"]},
)

z_positions = np.array(data.primary_sweep_values, dtype=float)
force_z = np.array(data.data_real("Force_on_Magnet.Force_z", convert_to_SI=True), dtype=float)

df = pd.DataFrame({"magnet_z0_cm": z_positions, "Force_z_N": force_z})
df = df.sort_values("magnet_z0_cm").reset_index(drop=True)
df

In [ ]:
## Export result data ##

csv_name = f"{proj_name}_force_result.csv"
csv_path = csv_name   # final result -- saved flat, next to the notebook
df.to_csv(csv_path, index=False)

In [ ]:
## Save project ##

M3D.save_project()